# tinylm pretrain — data source inspection

Stream-peeks every source in the pretrain pool — the live 5-way blend
(`tt` tiny-textbooks · `str` tiny-strange-textbooks · `fw` fineweb-edu ·
`ts` tinystories-v2 · `wt` tiny-webtext) plus wired-but-unused `cos` (cosmopedia-v2) —
so you can compare register, schema, document length, and how the blended 16k
tokenizer compresses each. Streaming means **no full download** (Cosmopedia isn't
downloaded; only the peeked rows are fetched).

In [4]:
import itertools
import os
import textwrap

os.environ.setdefault("HF_HOME", "/mnt/ai/data/hf")  # datasets cache

import numpy as np
import pandas as pd
from datasets import load_dataset

from chimera.data import (
    CosmopediaV2DataModule,
    FineWebEduTextDataModule,
    TinyStoriesV2DataModule,
    TinyStrangeTextbooksDataModule,
    TinyTextbooksDataModule,
    TinyWebTextDataModule,
)

WIDTH = 80

# Every source in the pretrain pool, keyed by the id used in the README Results
# table. The live 5-way blend is tt/str/fw/ts/wt; cos (Cosmopedia v2) is wired
# but not yet in a logged run.
SOURCES = {
    "tt": TinyTextbooksDataModule(data_dir="/mnt/ai/data"),
    "str": TinyStrangeTextbooksDataModule(data_dir="/mnt/ai/data"),
    "fw": FineWebEduTextDataModule(data_dir="/mnt/ai/data"),
    "ts": TinyStoriesV2DataModule(data_dir="/mnt/ai/data"),
    "wt": TinyWebTextDataModule(data_dir="/mnt/ai/data"),
    "cos": CosmopediaV2DataModule(data_dir="/mnt/ai/data"),
}


def wrap(text, width=WIDTH):
    """Soft-wrap each line to `width` cols while KEEPING the text's own newlines.

    Wrapping is done per line (split on the real newlines, wrap each, rejoin), so
    paragraph breaks / blank lines / structure are preserved — only over-long
    lines get folded. Long unbreakable tokens (e.g. URLs) are left intact.
    """
    return "\n".join(
        textwrap.fill(line, width=width, break_long_words=False, break_on_hyphens=False)
        if line.strip()
        else line
        for line in text.splitlines()
    )


def peek(dm, n=200):
    """Stream the first n raw rows of a source's train split — no full download.

    Reuses the module's own HF repo / config / shard bound (DATA_FILES), so what
    you inspect is exactly what pretraining draws from. Streaming means even the
    undownloaded Cosmopedia shards only fetch the rows n needs.
    """
    kw = {"streaming": True, "split": dm.TRAIN_SPLIT}
    if dm.DATA_FILES is not None:
        kw["data_files"] = dm.DATA_FILES
    if dm.CONFIG_NAME is not None:
        kw["name"] = dm.CONFIG_NAME
    return list(itertools.islice(load_dataset(dm.HF_REPO, **kw), n))

In [5]:
# Pull one sample per source once (streamed); every cell below reuses this.
N = 200
rows_by = {sid: peek(dm, N) for sid, dm in SOURCES.items()}
{sid: len(v) for sid, v in rows_by.items()}

Resolving data files:   0%|          | 0/69 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/69 [00:00<?, ?it/s]

{'tt': 200, 'str': 200, 'fw': 200, 'ts': 200, 'wt': 200, 'cos': 200}

## Schema + a sample document per source

In [10]:
# Schema + the first document of each source — see the register/format at a glance.
for sid, dm in SOURCES.items():
    r = rows_by[sid][0]
    print(f"{'=' * WIDTH}\n[{sid}]  {dm.HF_REPO}")
    print(f"  columns      : {list(r.keys())}")
    print(f"  text field(s): {dm.TEXT_COLUMNS or dm.TEXT_COLUMN}")
    print("-" * WIDTH)
    print(wrap(dm._row_text(r)[:2000] + ("..." if len(dm._row_text(r)) > 2000 else "")))
    print()

[tt]  nampdn-ai/tiny-textbooks
  columns      : ['text', 'source', 's', 'len', 'idx', 'textbook']
  text field(s): textbook
--------------------------------------------------------------------------------
Lesson: Understanding Government Finance

Introduction:
Governments around the world use various methods to finance their expenditures.
One such method is deficit financing, which involves borrowing money to fund
expenses that exceed income. In this lesson, we will learn about deficit
financing, its advantages and disadvantages, and how it works in practice.

Section 1: What is Deficit Financing?

Deficit financing refers to the practice of funding government expenditures by
borrowing money. This is typically done by issuing securities, such as treasury
bonds or notes, which are sold to investors. The proceeds from these sales are
then used to finance government spending, such as infrastructure projects,
social programs, and military operations.

Section 2: Advantages of Deficit Finan

## Length distribution + tokenizer compression

In [4]:
# Document-length distribution (one document = one training example, pre-packing).
recs = []
for sid, dm in SOURCES.items():
    lens = np.array([len(dm._row_text(r)) for r in rows_by[sid]])
    recs.append(
        {
            "id": sid,
            "repo": dm.HF_REPO.split("/")[-1],
            "n": len(lens),
            "chars_mean": int(lens.mean()),
            "chars_med": int(np.median(lens)),
            "chars_p10": int(np.percentile(lens, 10)),
            "chars_p90": int(np.percentile(lens, 90)),
            "chars_max": int(lens.max()),
        }
    )
pd.DataFrame(recs).set_index("id")

,repo,n,chars_mean,chars_med,chars_p10,chars_p90,chars_max
id,,,,,,,
tt,tiny-textbooks,200,2987,2984,2050,3954,5605
str,tiny-strange-textbooks,200,2781,2514,1561,4137,12880
fw,fineweb-edu,200,3877,2400,844,8091,65092
ts,TinyStoriesV2,200,812,744,561,1158,1956
wt,tiny-webtext,200,1381,1327,925,1936,2680
cos,smollm-corpus,200,3715,3455,2385,5479,8061


In [5]:
# How well the blended 16k vocab compresses each register (chars/token — higher
# is better). This is the same mixture-trained tokenizer pretraining uses, so a
# low chars/token flags a register the vocab serves poorly (wasted context).
from glob import glob

from chimera.tokenizers import BPETokenizer

tok_path = sorted(glob("/mnt/ai/data/mixture_tokenizers/tok_hf_v16384_*.json"))[-1]
tok = BPETokenizer.load(tok_path, backend="hf")
print("tokenizer:", tok_path.split("/")[-1], "\n")

recs = []
for sid, dm in SOURCES.items():
    texts = [dm._row_text(r) for r in rows_by[sid]]
    n_chars = sum(len(t) for t in texts)
    n_tokens = sum(len(ids) for ids in tok.encode_batch(texts))
    recs.append(
        {
            "id": sid,
            "chars/token": round(n_chars / n_tokens, 2),
            "avg_tokens/doc": round(n_tokens / len(texts)),
        }
    )
pd.DataFrame(recs).set_index("id")

tokenizer: tok_hf_v16384_c1000000000_6df93c68260a.json 



,chars/token,avg_tokens/doc
id,,
tt,4.61,648
str,4.56,610
fw,4.23,917
ts,4.04,201
wt,4.25,325
cos,4.67,796


## Browse full documents

In [9]:
# Eyeball full documents: show(source_id, index). Bump index to browse the sample.
def show(sid, i=0, maxchars=4000):
    wrap = lambda text: text

    dm = SOURCES[sid]
    print(f"[{sid}]  {dm.HF_REPO}  ·  doc #{i}\n{'-' * WIDTH}")
    print(wrap(dm._row_text(rows_by[sid][i])[:maxchars]))


show("fw", 0)  # tiny-webtext: note the human/bot Q&A structure joined into one doc

[fw]  HuggingFaceFW/fineweb-edu  ·  doc #0
--------------------------------------------------------------------------------
The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.
The freedom she exhibited in finally accepting him in direct defiance of Lady Catherine and knowing her father would disapprove was unusual even for Austen. In her last book Anne Elliot is persuaded to refuse Captain Wentworth at Lady Russel’s insistence.
Although Jane played by the rules of the day, all of her writing is infused with how she wanted life to be. She ‘screams’ her outrage at the limitations for women in Emma.
When accosted by Mrs. Elton, J

In [11]:
show("fw", 1, maxchars=-1)

[fw]  HuggingFaceFW/fineweb-edu  ·  doc #1
--------------------------------------------------------------------------------
Taking Play Seriously
By ROBIN MARANTZ HENIG
Published: February 17, 2008
On a drizzly Tuesday night in late January, 200 people came out to hear a psychiatrist talk rhapsodically about play -- not just the intense, joyous play of children, but play for all people, at all ages, at all times. (All species too; the lecture featured touching photos of a polar bear and a husky engaging playfully at a snowy outpost in northern Canada.) Stuart Brown, president of the National Institute for Play, was speaking at the New York Public Library's main branch on 42nd Street. He created the institute in 1996, after more than 20 years of psychiatric practice and research persuaded him of the dangerous long-term consequences of play deprivation. In a sold-out talk at the library, he and Krista Tippett, host of the public-radio program ''Speaking of Faith,'' discussed the biologic

In [6]:
import asyncio
import json

from ollama import AsyncClient

MODEL = "gpt-oss:latest"

SYSTEM_PROMPT = """You are an expert data annotator labeling webpage excerpts from the FineWeb-Edu dataset.

For each excerpt, you must:

1. Write a concise summary of its main topic and useful information.
2. Assign between 1 and 4 tags from the controlled vocabulary below.

## Output format

Return exactly one valid JSON object:

{"summary":"Concise summary of the excerpt.","tags":["Primary Tag","Optional Tag"]}

## Output rules

* Output valid JSON only.
* Do not use Markdown, code fences, comments, or additional text.
* Include exactly two keys: `"summary"` and `"tags"`.
* Write the summary in 1-2 complete sentences, usually 20-60 words.
* Describe the actual content rather than saying “the excerpt discusses” or “the webpage explains.”
* Base the summary and tags only on information supported by the provided text.
* Do not invent missing context, authorship, sources, conclusions, or factual details.
* Ignore navigation text, advertisements, cookie notices, and unrelated boilerplate when meaningful content is present.
* Use between 1 and 4 unique tags.
* Use tag names exactly as written below.
* Order tags from most relevant to least relevant.
* The first tag must be the excerpt's primary subject.
* Prefer a specific subject tag over a broad or loosely related tag.
* Use no more than three subject tags and no more than one format or intent tag.
* Do not assign a tag merely because a related word appears once.
* When the subject cannot be determined reliably, use `"Other / Unclear"`.
* When the excerpt contains almost no meaningful information, use `"Low Information"` as its format tag.

## Subject tags

* Mathematics
* Statistics
* Computer Science
* Engineering & Technology
* Physics
* Chemistry
* Biology
* Earth & Environmental Science
* Medicine & Health
* Psychology
* Economics
* Finance
* Business & Management
* Law
* Politics & Government
* History
* Geography
* Sociology & Anthropology
* Philosophy & Ethics
* Religion
* Literature
* Language & Linguistics
* Education
* Art & Design
* Music
* Media & Communication
* Agriculture & Food
* Sports & Fitness
* Career & Professional Development
* General Reference
* Other / Unclear

## Format and intent tags

* Tutorial / How-To
* Explanatory
* Reference
* Academic / Research
* Argumentative / Opinion
* News / Current Events
* Question & Answer
* Personal Narrative
* Creative Writing
* Commercial / Promotional
* Low Information

## Tag-selection guidance

* Use `"Tutorial / How-To"` for procedural, step-by-step, or instructional content.
* Use `"Explanatory"` for content primarily teaching or clarifying a concept.
* Use `"Reference"` for definitions, factual listings, documentation, encyclopedic entries, or lookup-oriented material.
* Use `"Academic / Research"` for scholarly analysis, research findings, formal methodology, or technical papers.
* Use `"Argumentative / Opinion"` when the author advances or defends a position.
* Use `"News / Current Events"` for reporting centered on a recent event or development.
* Use `"Question & Answer"` for interviews, FAQs, forum answers, or direct responses to questions.
* Use `"Personal Narrative"` for autobiographical experiences or personal reflections.
* Use `"Creative Writing"` for fiction, poetry, scripts, or other imaginative writing.
* Use `"Commercial / Promotional"` for sales-oriented, marketing, product-promotion, or lead-generation content.
* Use `"Low Information"` for fragments dominated by menus, metadata, broken text, boilerplate, or content too limited to summarize meaningfully.

## Examples

Input:
Python dictionaries store key-value pairs. Values can be retrieved by placing a key inside square brackets. The get method can instead return a default value when the key is missing.

Output:
{"summary":"Python dictionaries associate keys with values and support direct lookup with square brackets. The get method provides a safer lookup option by returning a default when a key is absent.","tags":["Computer Science","Tutorial / How-To"]}

Input:
The author argues that raising interest rates too quickly can reduce investment and employment, even when the policy successfully lowers inflation.

Output:
{"summary":"The author argues that rapid interest-rate increases may lower inflation but can also reduce investment and employment.","tags":["Economics","Finance","Argumentative / Opinion"]}

Input:
Home | About | Subscribe | Privacy Policy | Copyright 2024

Output:
{"summary":"The excerpt contains navigation and website boilerplate without enough substantive content to identify a topic.","tags":["Other / Unclear","Low Information"]}
"""


async def annotate(client, document):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Annotate the following document:\n\n<document>{document}</document>",
        },
    ]
    response = await client.chat(model=MODEL, messages=messages)
    return response.message.content


async def run(documents):
    client = AsyncClient()
    return await asyncio.gather(*(annotate(client, doc) for doc in documents))


# Annotate the first 20 FineWeb-Edu documents from the streamed sample.
documents = [SOURCES["fw"]._row_text(r) for r in rows_by["fw"][:20]]
outputs = await run(documents)  # Jupyter runs its own event loop; await directly

for i, (doc, out) in enumerate(zip(documents, outputs)):
    print(f"{'=' * WIDTH}\n[{i}]  {doc[:80].strip()}...")
    print("-" * WIDTH)
    try:
        print(json.dumps(json.loads(out), indent=2, ensure_ascii=False))
    except json.JSONDecodeError:
        print(out)
    print()

[0]  The Independent Jane
For all the love, romance and scandal in Jane Austen’s book...
--------------------------------------------------------------------------------
{
  "summary": "The passage analyzes Jane Austen’s novels as promoting themes of personal freedom and independent thought, drawing parallels between her advocacy for women’s autonomy and the principles of American independence articulated by figures such as Jefferson and Adams.",
  "tags": [
    "Literature",
    "Explanatory"
  ]
}

[1]  Taking Play Seriously
By ROBIN MARANTZ HENIG
Published: February 17, 2008
On a d...
--------------------------------------------------------------------------------
{
  "summary": "An overview of a 200‑person lecture by psychiatrist Stuart Brown that argues play is essential for human and animal development, linking it to neurological growth, health, and social skills; the piece discusses modern anxieties over reduced free play, digital alternatives, and the emerging scientific study 

In [ ]:
print("hi")

hi
